In [ ]:
pip install -U transformers accelerate bitsandbytes peft trl datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 147.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 465.5/465.5 kB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.6/511.6 kB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 56.2 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.2
    Uninstalling transformers-4.57.2:
      Successfully uninstalled transformers-4.57.2
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


# Import Libraries

In [ ]:
import os, json, torch
import pandas as pd
from datasets import Dataset
from collections import Counter
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    BitsAndBytesConfig, LogitsProcessor, LogitsProcessorList
)
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig, PeftModel, TaskType

import os
os.environ["WANDB_DISABLED"] = "true"


# Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Configuration

In [ ]:
Model = "mistralai/Mistral-7B-Instruct-v0.3"
BASE_DIR = "/content/drive/MyDrive/Colab Notebooks/DataSci_266_NLP/Final_Project"
OUTPUT_DIR = BASE_DIR + "/Medical_Abstract/mistral_cls_adapter_001"
os.makedirs(OUTPUT_DIR, exist_ok=True)

Train_path = BASE_DIR + "/Medical_Abstract/train_medical_abstract.csv"
Val_path = BASE_DIR + "/Medical_Abstract/val_medical_abstract.csv"
Test_path = BASE_DIR + "/Medical_Abstract/test_medical_abstract.csv"

Labels = ["Neoplasms", "Digestive system diseases", "Nervous system diseases", "Cardiovascular diseases", "General pathological conditions"]

In [ ]:
train_df = pd.read_csv(Train_path, dtype={"text":str, "label":str})
display(train_df)

,text,label
0,Reoperations on heart valve prostheses: an ana...,4
1,Malnutrition and carbohydrate malabsorption in...,1
2,Complete follow-up and evaluation of a skin ca...,0
3,Gastrointestinal motor dysfunction in acquired...,2
4,Effect of sleep-induced increases in upper air...,4
...,...,...
6606,Biliary and pancreatic metastases of breast ca...,1
6607,Itraconazole therapy in aspergillosis: study i...,4
6608,Dreams and epilepsy. The relationship between ...,2
6609,Ear involvement in the yellow nail syndrome. R...,4


# Data Preparation

In [ ]:
def build_example(text, label=None):
  try:
    system = "You are a helpful classifier. Reply with exactly one label from: " + ", ".join(Labels) + "."
    user = f"Text: {text}\nLabel options: " + ", ".join(Labels) + "\nAnswer with one label only."
    return {
        "system": system,
        "user": user,
        "assistant": label or "",
    }
  except:
    print(text, "\n")
    raise

recs = []
def df_to_hf(df, is_train=True): # converts df into a Hugging Face dataset
    recs = []
    try:
      for _, row in df.iterrows():
          recs.append(build_example(row["text"], Labels[row["label"]] if is_train else None))
      return Dataset.from_list(recs)
    except:
      print(row, "\n")
      # return Dataset.from_list(recs)


train_df = pd.read_csv(Train_path)
val_df = pd.read_csv(Val_path)

train_ds = df_to_hf(train_df)
val_ds = df_to_hf(val_df)

In [ ]:
train_ds

Dataset({
    features: ['system', 'user', 'assistant'],
    num_rows: 6611
})

In [ ]:
print(train_ds[:2]["assistant"])

['General pathological conditions', 'Digestive system diseases']


In [ ]:
print(f"[Step 2] Train rows={len(train_df):,}, Val rows={len(val_df):,}")
print(f"[Step 2] Label examples: {train_df['label'].value_counts().to_dict()}")

[Step 2] Train rows=6,611, Val rows=2,834
[Step 2] Label examples: {4: 2119, 0: 1543, 3: 1445, 2: 848, 1: 656}


# Tokenizer + Quantization

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(Model)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

# LoRA Configuration

In [ ]:
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "down_proj", "up_proj"],
)

train_cfg = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    eval_strategy="steps",
    eval_steps=500,
    logging_steps=50,
    save_steps=1000,
    bf16=True,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    packing=False,
    dataset_num_proc=4,
    report_to="none",
    logging_dir="./logs",
)


# Training

In [ ]:
def formatting_func(example):
    messages = [
        {"role": "system", "content": example["system"]},
        {"role": "user", "content": example["user"]},
        {"role": "assistant", "content": example["assistant"]},
    ]
    # Convert chat messages to a single text sequence (no tokenization yet)
    return tokenizer.apply_chat_template(messages, tokenize=False)

trainer = SFTTrainer(
    model=Model,                 # or model name string
    peft_config=peft_config,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=train_cfg,
    formatting_func=formatting_func,  # single-example formatter
)

config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.55G [00:00<?, ?B/s]

...

Truncating train dataset (num_proc=4):   0%|          | 0/6611 [00:00<?, ? examples/s]

Applying formatting function to eval dataset (num_proc=4):   0%|          | 0/2834 [00:00<?, ? examples/s]

Adding EOS to eval dataset (num_proc=4):   0%|          | 0/2834 [00:00<?, ? examples/s]

Tokenizing eval dataset (num_proc=4):   0%|          | 0/2834 [00:00<?, ? examples/s]

Truncating eval dataset (num_proc=4):   0%|          | 0/2834 [00:00<?, ? examples/s]

In [ ]:
trainer.train()
trainer.save_model()
tokenizer.save_pretrained(OUTPUT_DIR)

Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
500,1.073700,1.334088,1.172765,2893611.000000,0.682597


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


('/content/drive/MyDrive/Colab Notebooks/DataSci_266_NLP/Final_Project/Helen/mistral_cls_adapter_001/tokenizer_config.json',
 '/content/drive/MyDrive/Colab Notebooks/DataSci_266_NLP/Final_Project/Helen/mistral_cls_adapter_001/special_tokens_map.json',
 '/content/drive/MyDrive/Colab Notebooks/DataSci_266_NLP/Final_Project/Helen/mistral_cls_adapter_001/chat_template.jinja',
 '/content/drive/MyDrive/Colab Notebooks/DataSci_266_NLP/Final_Project/Helen/mistral_cls_adapter_001/tokenizer.model',
 '/content/drive/MyDrive/Colab Notebooks/DataSci_266_NLP/Final_Project/Helen/mistral_cls_adapter_001/added_tokens.json',
 '/content/drive/MyDrive/Colab Notebooks/DataSci_266_NLP/Final_Project/Helen/mistral_cls_adapter_001/tokenizer.json')